In [3]:
import pandas as pd, numpy as np
from sklearn.ensemble import IsolationForest
w=pd.read_csv('master_works.csv')
d=w[w.sanction_amount.notna()].copy()
print('works with sanction amount:',len(d))


works with sanction amount: 36000


In [4]:
# group sizes
g=d.groupby('work_type').size().sort_values(ascending=False)
print('work types:',len(g),'| types with >=30 works:',(g>=30).sum(),'covering',round(g[g>=30].sum()/len(d),3))
# robust z within work_type (>=30), else state-level fallback


work types: 105 | types with >=30 works: 58 covering 0.988


In [5]:
def rz(s):
    m=s.median(); mad=(s-m).abs().median()
    return (s-m)/(1.4826*mad) if mad>0 else s*0
d['z']=np.nan
big=d.work_type.map(g)>=30
d.loc[big,'z']=d[big].groupby('work_type').log_amount.transform(rz)
d['z_flag']=(d.z.abs()>3.5)
print('z flagged:',int(d.z_flag.sum()),'of',int(big.sum()))


z flagged: 738 of 35571


In [6]:
# isolation forest
feat=['log_amount','days_rec_to_sanction','round_amount']
X=d[feat].copy()
X['type_med']=d.groupby('work_type').log_amount.transform('median')
X['dev']=d.log_amount-X.type_med
X['state_code']=d.state.astype('category').cat.codes
X=X.fillna(X.median())
iso=IsolationForest(n_estimators=200,contamination=0.02,random_state=42).fit(X)
d['if_flag']=iso.predict(X)==-1
print('IF flagged:',int(d.if_flag.sum()))
both=(d.z_flag&d.if_flag).sum();print('overlap',both,'| z only',int((d.z_flag&~d.if_flag).sum()),'| IF only',int((~d.z_flag&d.if_flag).sum()))


IF flagged: 720
overlap 130 | z only 608 | IF only 590


In [7]:
# synthetic injection recall
rng=np.random.default_rng(0)
idx=rng.choice(d.index[big],500,replace=False)
t=d.copy(); t.loc[idx,'sanction_amount']*=5
t['log_amount']=np.log1p(t.sanction_amount)
t['zz']=t[big].groupby('work_type').log_amount.transform(rz)
zr=(t.loc[idx,'zz'].abs()>3.5).mean()
X2=t[feat].copy();X2['type_med']=t.groupby('work_type').log_amount.transform('median');X2['dev']=t.log_amount-X2.type_med;X2['state_code']=t.state.astype('category').cat.codes;X2=X2.fillna(X2.median())
ir=(iso.predict(X2.loc[idx])==-1).mean()
print('recall on x5 injected: z',round(zr,3),'IF',round(ir,3))


recall on x5 injected: z 0.19 IF 0.22


In [11]:
import pandas as pd

# 1. Define Boolean masks to isolate mutually exclusive outlier subgroups.
# This helps us study 'pure' profiles (anomalies caught by only one specific method versus both).
z_only_mask = d['z_flag'] & ~d['if_flag']      # Outliers flagged ONLY by Robust Z-Score
if_only_mask = ~d['z_flag'] & d['if_flag']     # Outliers flagged ONLY by Isolation Forest
both_mask = d['z_flag'] & d['if_flag']         # Outliers flagged by BOTH methodologies

# 2. Select key features to analyze across the different flagged groups.
# This list contains physical values, logarithmic distributions, timelines, and discrete flags.
comparison_features = ['log_amount', 'sanction_amount', 'days_rec_to_sanction', 'round_amount']

# 3. Calculate median and mean metrics for each distinct group.
# This step exposes the structural differences (e.g., scale of money, delays, and rounding behavior)
# which explain why each model flagged these specific records.
summary_data = []
for group_name, mask in [('Z-Score Only', z_only_mask), ('Isolation Forest Only', if_only_mask), ('Both Flagged', both_mask)]:
    sub_df = d[mask]
    summary = {
        'Group': group_name,
        'Count': len(sub_df),
        'Median Log Amount': sub_df['log_amount'].median(),         # Reveals the characteristic scale of the budget in log space
        'Mean Sanction Amount': sub_df['sanction_amount'].mean(),   # Exposes the actual financial impact/scale of the projects
        'Median Days to Sanction': sub_df['days_rec_to_sanction'].median(), # Shows if processing delay is a distinguishing outlier feature
        'Mean Round Amount Flag': sub_df['round_amount'].mean()    # Shows the ratio of rounded vs. unrounded numbers in the group
    }
    summary_data.append(summary)

# 4. Convert the collected statistics into a structured DataFrame and display it.
# This provides a clear side-by-side table to easily compare and contrast the outlier profiles.
summary_df = pd.DataFrame(summary_data)
display(summary_df)

,Group,Count,Median Log Amount,Mean Sanction Amount,Median Days to Sanction,Mean Round Amount Flag
0,Z-Score Only,608,11.289794,4.843139e+05,101.5,0.064145
1,Isolation Forest Only,590,14.508658,2.461852e+06,376.5,0.684746
2,Both Flagged,130,15.424949,6.854819e+06,140.0,0.376923
